[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/091.%20The%20Contract%20Design%20to%20Mitigate%20the%20Bullwhip%20Effect/P91-Tier-7_executed.ipynb)

---



# P91-Tier-7: Human-AI Symbiotic Partnership

## Purpose and scope

This notebook simulates a sequence of contract-design sessions between a supply-chain manager and an explainable, rule-based AI recommender. The AI analyzes demand statistics and proposes contract parameters; the human applies explicit experience-based rules; the partnership updates a trust score and lets the AI incorporate human modifications into later recommendations.

The inputs are a demand time series, a small set of partner and data-quality indicators, and fixed human and AI settings. The outputs are session-level recommendations, acceptance and trust scores, simplified performance measures, plots, and a comparison with AI-only and human-only heuristics. The code is a transparent teaching simulation: it does not train a machine-learning model, solve a mathematical optimization problem, or establish that collaboration causes better real-world outcomes.

## Modeling assumptions

- Demand is represented by a finite sequence of nonnegative observations. The generated example uses 52 periods per scenario.
- Contract parameters are dimensionless teaching quantities or simplified monetary inputs; the simulation does not define a complete real-world contract or calibrate parameters to a particular industry.
- Partner reliability, flexibility, and data quality are fixed scenario descriptors. The human uses reliability and flexibility; the AI confidence calculation uses data quality.
- Human behavior is represented by explicit threshold rules, not by a psychological or behavioral model. Trust is a bounded score, not a measured probability.
- The performance calculation is a stylized proxy for order amplification, cost, service level, and profit. It omits such factors as capacity, lead-time dynamics, inventory balance, taxes, and strategic interactions among supply-chain tiers.
- The comparison methods are illustrative heuristics evaluated with the same performance proxy. Their results depend on the stated rules and inputs and should not be interpreted as a general ranking of human and AI decision-making.

## Collaboration framework

The session follows this order: demand analysis, AI recommendation, human evaluation and modification, trust update, AI preference learning, and performance assessment. This order makes the feedback loop explicit: modifications in one session can influence the AI recommendation in a later session.

The AI component summarizes demand patterns and applies fixed recommendation rules. The human component applies relationship-, risk-, and experience-based adjustments. The symbiotic component records both contributions and passes the human's parameter changes back to the AI's simple preference-learning rule.

## Mathematical representation of the implemented rules

Let \(a_j\) be the AI recommendation and \(h_j\) the human-modified value for parameter \(j\). The implementation measures average relative modification and defines the acceptance score as

$$
\bar{d} = \frac{1}{m}\sum_{j=1}^{m}\frac{|h_j-a_j|}{|a_j|},
\qquad
A = \max(0, 1 - 2\bar{d})(0.5 + 0.5C),
$$

where \(m\) is the number of contract parameters and \(C\in[0,1]\) is AI confidence. All AI parameter values in this example are positive, so the relative differences are defined. The code implements this calculation in `HumanExpert._calculate_acceptance_score`.

Trust \(T\) is updated using confidence and acceptance thresholds:

$$
T_{t+1} =
\begin{cases}
T_t + \alpha(1-T_t), & C_t > 0.7 \text{ and } A_t > 0.7,\\
T_t - \beta T_t, & C_t > 0.7 \text{ and } A_t < 0.3,\\
T_t, & \text{otherwise},
\end{cases}
\qquad
T_{t+1} \leftarrow \min(0.95,\max(0.1,T_{t+1})).
$$

Here \(t\) indexes sessions, and \(\alpha\) and \(\beta\) are the positive and negative trust-update rates. In the code, these are `trust_alpha` and `trust_beta`, and the update is performed by `SymbioticPartnership._update_trust`. Acceptance is used as a proxy for a favorable interaction; the model does not observe whether a recommendation was objectively successful.

The AI's preference update is also a simple rule. When acceptance is below 0.5, a sufficiently large human-to-AI parameter ratio \(r_j=h_j/a_j\) updates the stored preference factor \(p_j\) as

$$
p_j \leftarrow (1-\eta)p_j + \eta r_j,
$$

where \(\eta\) is the learning rate. The code applies this rule in `ExplainableAI.learn_from_human`; the learned factors multiply the corresponding parameters in subsequent recommendations. This is a deliberately simple feedback mechanism, not a fitted learning model.

For performance assessment, the simulation uses demand standard deviation \(\sigma_D\), mean \(\mu_D\), flexibility \(f\), and safety stock \(s\):

$$
g=(2-f)\frac{s}{1.5},\qquad
\text{Bullwhip Ratio}=\frac{\sigma_D g}{\sigma_D+10^{-6}},
$$

$$
\text{Holding Cost}=0.2\mu_Ds,\qquad
\text{Stockout Cost}=0.1\sigma_Dg,
$$

$$
\text{Net Profit}=100\mu_Dq+0.5b-\text{Holding Cost}-\text{Stockout Cost},
\qquad
\text{Service Level}=\max\left(0,1-\frac{\text{Stockout Cost}}{10\mu_D}\right),
$$

where \(g\) is the order-amplification factor, \(q\) is revenue sharing, and \(b\) is the information-sharing bonus. These equations are implemented by `SymbioticPartnership._simulate_contract_performance`. The small constant in the bullwhip denominator prevents division by zero when demand has no variation; the generated demand has a positive mean, as required by the service-level calculation.

Demand statistics are computed in one pass over each time series, so their work grows linearly with the number of observations. With a fixed number of contract parameters, each session's recommendation, evaluation, and performance calculations also grow linearly with the demand-series length. Across \(S\) sessions of length \(N\), the principal time cost is \(O(SN)\); the stored session history grows with the number of sessions.

## Concrete example

The example uses a fictional supply-chain director with 12 years of experience and four generated demand scenarios: stable, volatile, growing, and declining. The scenarios support a demonstration of the workflow; they are not observations from a real electronics supply chain. Run the notebook from top to bottom to generate the session results and visualizations. No numerical outcome is asserted in advance because the displayed results are produced by the code.

### Implementation sequence

1. Define the human expert, AI recommender, and partnership workflow.
2. Initialize the example profiles and fixed context.
3. Generate reproducible demand scenarios.
4. Conduct one collaborative session for each scenario.
5. Summarize trust, acceptance, and simulated performance.
6. Visualize the session outputs and compare the stated heuristic approaches.


In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class HumanExpert:
    """Represent the human expert who reviews AI contract recommendations."""

    name: str
    experience_years: int
    expertise_areas: List[str]
    risk_tolerance: float
    relationship_focus: float
    initial_trust: float

    def evaluate_contract(
        self,
        ai_params: Dict,
        ai_confidence: float,
        context: Dict,
    ) -> Tuple[Dict, float, str]:
        """Apply explicit human decision rules and score AI acceptance."""
        human_params = ai_params.copy()
        modifications = []

        if self.relationship_focus > 0.5:
            partner_history = context.get("partner_history", {})
            if partner_history.get("reliability", 0.5) < 0.7:
                human_params["safety_stock"] *= 1.2
                modifications.append(
                    "Increased safety stock for a less reliable partner"
                )

            if partner_history.get("flexibility", 0.5) > 0.7:
                human_params["flexibility_factor"] *= 1.1
                modifications.append(
                    "Increased flexibility for a cooperative partner"
                )

        if self.risk_tolerance < 0.3:
            human_params["revenue_sharing"] *= 0.95
            modifications.append(
                "Applied conservative revenue sharing because of risk aversion"
            )
        elif self.risk_tolerance > 0.7:
            human_params["lead_time_buffer"] *= 0.9
            modifications.append(
                "Reduced the lead-time buffer for an aggressive strategy"
            )

        if self.experience_years > 10:
            human_params["info_sharing_bonus"] *= 1.15
            modifications.append(
                "Enhanced information sharing based on experience"
            )

        acceptance_score = self._calculate_acceptance_score(
            ai_params, human_params, ai_confidence, context
        )

        reasoning = (
            f"Human expert {self.name} ({self.experience_years} years of "
            f"experience) made {len(modifications)} modifications: "
            + "; ".join(modifications)
        )

        return human_params, acceptance_score, reasoning

    def _calculate_acceptance_score(
        self,
        ai_params: Dict,
        human_params: Dict,
        ai_confidence: float,
        context: Dict,
    ) -> float:
        """Score acceptance from relative parameter changes and confidence."""
        if not ai_params:
            raise ValueError("The AI recommendation must contain parameters.")

        relative_differences = []
        for key, ai_value in ai_params.items():
            if key not in human_params:
                raise ValueError(f"Human parameters are missing {key!r}.")
            if ai_value == 0:
                raise ValueError(
                    f"Relative differences require a nonzero AI value for {key!r}."
                )
            relative_differences.append(
                abs(human_params[key] - ai_value) / abs(ai_value)
            )

        average_difference = float(np.mean(relative_differences))
        acceptance = max(0.0, 1.0 - 2.0 * average_difference)
        acceptance *= 0.5 + 0.5 * ai_confidence
        return float(acceptance)

In [ ]:
class ExplainableAI:
    """Provide rule-based demand analysis and contract recommendations."""

    def __init__(self, model_complexity: str = "advanced"):
        self.model_complexity = model_complexity
        self.confidence_threshold = 0.7
        self.learning_rate = 0.1
        self.human_preferences = {}

    def analyze_demand_patterns(self, demand_data: np.ndarray) -> Dict:
        """Summarize demand variability, trend, and simple seasonality."""
        demand_data = np.asarray(demand_data, dtype=float)
        if demand_data.ndim != 1 or demand_data.size < 2:
            raise ValueError("Demand data must be a one-dimensional series of length at least two.")
        if not np.all(np.isfinite(demand_data)) or np.any(demand_data < 0):
            raise ValueError("Demand observations must be finite and nonnegative.")

        mean_demand = float(np.mean(demand_data))
        standard_deviation = float(np.std(demand_data))
        coefficient_of_variation = (
            standard_deviation / mean_demand if mean_demand > 0 else 0.0
        )
        stats = {
            "mean": mean_demand,
            "std": standard_deviation,
            "cv": coefficient_of_variation,
            "trend": self._calculate_trend(demand_data),
            "seasonality": self._detect_seasonality(demand_data),
        }

        insights = []
        if stats["cv"] > 0.3:
            insights.append(
                "High demand variability detected; consider flexible contracts"
            )
        if abs(stats["trend"]) > 0.1:
            direction = "Increasing" if stats["trend"] > 0 else "Decreasing"
            insights.append(f"{direction} association with time detected")
        if stats["seasonality"] > 0.2:
            insights.append(
                "Lagged demand correlations detected; consider seasonal contracts"
            )

        return {
            "statistics": stats,
            "insights": insights,
            "complexity_score": self._assess_complexity(stats),
        }

    def generate_contract_recommendation(
        self,
        demand_analysis: Dict,
        context: Dict,
    ) -> Tuple[Dict, float, List[str]]:
        """Apply transparent rules to produce parameters and confidence."""
        stats = demand_analysis["statistics"]
        params = {
            "revenue_sharing": 0.5,
            "flexibility_factor": 1.0,
            "safety_stock": 1.5,
            "info_sharing_bonus": 50,
            "lead_time_buffer": 0.5,
        }
        reasoning = []

        if stats["cv"] > 0.3:
            params["flexibility_factor"] = 1.2
            params["safety_stock"] = 2.0
            reasoning.append(
                "Increased flexibility and safety stock for high variability"
            )

        if stats["trend"] > 0.1:
            params["revenue_sharing"] = 0.55
            reasoning.append("Adjusted revenue sharing for an increasing trend")
        elif stats["trend"] < -0.1:
            params["revenue_sharing"] = 0.45
            reasoning.append("Adjusted revenue sharing for a decreasing trend")

        for param, adjustment in self.human_preferences.items():
            if param in params:
                params[param] *= adjustment
                reasoning.append(f"Applied learned preference for {param}")

        confidence = self._calculate_confidence(stats, context)
        return params, confidence, reasoning

    def learn_from_human(
        self,
        human_params: Dict,
        ai_params: Dict,
        acceptance_score: float,
    ) -> None:
        """Update preference factors after substantial human modification."""
        if acceptance_score < 0.5:
            for param, ai_value in ai_params.items():
                if param not in human_params or ai_value == 0:
                    continue

                ratio = human_params[param] / ai_value
                if abs(ratio - 1.0) > 0.1:
                    current_preference = self.human_preferences.get(param, 1.0)
                    self.human_preferences[param] = (
                        (1.0 - self.learning_rate) * current_preference
                        + self.learning_rate * ratio
                    )

    def _calculate_trend(self, demand_data: np.ndarray) -> float:
        """Return the correlation between time index and demand."""
        if np.std(demand_data) == 0:
            return 0.0
        time_index = np.arange(len(demand_data), dtype=float)
        correlation = np.corrcoef(time_index, demand_data)[0, 1]
        return float(correlation) if np.isfinite(correlation) else 0.0

    def _detect_seasonality(self, demand_data: np.ndarray) -> float:
        """Summarize absolute autocorrelation at selected time lags."""
        if len(demand_data) < 12:
            return 0.0

        seasonal_correlations = []
        for lag in (4, 8, 12):
            if len(demand_data) > lag:
                first = demand_data[:-lag]
                second = demand_data[lag:]
                if np.std(first) > 0 and np.std(second) > 0:
                    correlation = np.corrcoef(first, second)[0, 1]
                    if np.isfinite(correlation):
                        seasonal_correlations.append(abs(float(correlation)))

        return (
            float(np.mean(seasonal_correlations))
            if seasonal_correlations
            else 0.0
        )

    def _assess_complexity(self, stats: Dict) -> float:
        """Combine bounded variability, trend, and seasonality scores."""
        complexity = 0.0
        complexity += min(stats["cv"], 1.0) * 0.4
        complexity += min(abs(stats["trend"]), 1.0) * 0.3
        complexity += min(stats["seasonality"], 1.0) * 0.3
        return float(complexity)

    def _calculate_confidence(self, stats: Dict, context: Dict) -> float:
        """Scale a baseline confidence by complexity and data quality."""
        complexity = self._assess_complexity(stats)
        confidence = 0.8 * (1.0 - complexity * 0.3)
        data_quality = context.get("data_quality", 0.8)
        confidence *= data_quality

        if self.model_complexity == "advanced":
            confidence *= 1.1

        return float(min(0.95, max(0.5, confidence)))

In [ ]:
class SymbioticPartnership:
    """Coordinate human review, AI learning, trust, and simulation."""

    def __init__(self, human_expert: HumanExpert, ai_system: ExplainableAI):
        self.human = human_expert
        self.ai = ai_system
        self.trust_level = human_expert.initial_trust
        self.session_history = []
        self.performance_metrics = []
        self.trust_alpha = 0.1
        self.trust_beta = 0.05

    def conduct_collaborative_session(
        self,
        demand_data: np.ndarray,
        context: Dict,
    ) -> Dict:
        """Run one session in the stated collaboration sequence."""
        session_result = {
            "session_id": len(self.session_history) + 1,
            "demand_analysis": None,
            "ai_recommendation": None,
            "human_evaluation": None,
            "final_contract": None,
            "trust_before": self.trust_level,
            "trust_after": None,
            "performance": None,
        }

        demand_analysis = self.ai.analyze_demand_patterns(demand_data)
        session_result["demand_analysis"] = demand_analysis

        ai_params, ai_confidence, ai_reasoning = (
            self.ai.generate_contract_recommendation(demand_analysis, context)
        )
        session_result["ai_recommendation"] = {
            "parameters": ai_params,
            "confidence": ai_confidence,
            "reasoning": ai_reasoning,
        }

        human_params, acceptance_score, human_reasoning = (
            self.human.evaluate_contract(ai_params, ai_confidence, context)
        )
        session_result["human_evaluation"] = {
            "parameters": human_params,
            "acceptance_score": acceptance_score,
            "reasoning": human_reasoning,
        }

        self._update_trust(acceptance_score, ai_confidence)
        session_result["trust_after"] = self.trust_level

        self.ai.learn_from_human(human_params, ai_params, acceptance_score)

        final_contract = human_params
        session_result["final_contract"] = final_contract
        performance = self._simulate_contract_performance(
            final_contract, demand_data
        )
        session_result["performance"] = performance

        self.session_history.append(session_result)
        self.performance_metrics.append(performance)
        return session_result

    def _update_trust(
        self,
        acceptance_score: float,
        ai_confidence: float,
) -> None:
        """Apply bounded positive or negative trust reinforcement."""
        if ai_confidence > 0.7 and acceptance_score > 0.7:
            self.trust_level += self.trust_alpha * (1.0 - self.trust_level)
        elif ai_confidence > 0.7 and acceptance_score < 0.3:
            self.trust_level -= self.trust_beta * self.trust_level

        self.trust_level = max(0.1, min(0.95, self.trust_level))

    def _simulate_contract_performance(
        self,
        contract_params: Dict,
        demand_data: np.ndarray,
) -> Dict:
        """Calculate stylized performance indicators for a contract."""
        demand_data = np.asarray(demand_data, dtype=float)
        demand_std = float(np.std(demand_data))
        demand_mean = float(np.mean(demand_data))
        if demand_mean <= 0:
            raise ValueError("Mean demand must be positive for this simulation.")

        flexibility = contract_params.get("flexibility_factor", 1.0)
        safety_stock = contract_params.get("safety_stock", 1.5)
        order_amplification = (2.0 - flexibility) * (safety_stock / 1.5)
        order_std = demand_std * order_amplification
        bullwhip_ratio = order_std / (demand_std + 1e-6)

        holding_cost = demand_mean * safety_stock * 0.2
        stockout_cost = demand_std * order_amplification * 0.1
        revenue = (
            demand_mean
            * 100
            * contract_params.get("revenue_sharing", 0.5)
        )
        info_benefit = contract_params.get("info_sharing_bonus", 0) * 0.5
        total_cost = holding_cost + stockout_cost
        net_profit = revenue + info_benefit - total_cost
        service_level = max(0.0, 1.0 - stockout_cost / (demand_mean * 10.0))

        return {
            "bullwhip_ratio": float(bullwhip_ratio),
            "total_cost": float(total_cost),
            "net_profit": float(net_profit),
            "service_level": float(service_level),
        }

    def get_partnership_summary(self) -> Dict:
        """Aggregate session-level trust, acceptance, and performance."""
        if not self.session_history:
            return {"message": "No sessions conducted yet"}

        trust_evolution = [
            session["trust_before"] for session in self.session_history
        ]
        trust_evolution.append(self.trust_level)
        acceptance_scores = [
            session["human_evaluation"]["acceptance_score"]
            for session in self.session_history
        ]
        average_bullwhip = float(
            np.mean([metric["bullwhip_ratio"] for metric in self.performance_metrics])
        )
        average_profit = float(
            np.mean([metric["net_profit"] for metric in self.performance_metrics])
        )

        return {
            "total_sessions": len(self.session_history),
            "trust_evolution": trust_evolution,
            "final_trust_level": self.trust_level,
            "average_acceptance": float(np.mean(acceptance_scores)),
            "average_bullwhip_ratio": average_bullwhip,
            "average_profit": average_profit,
            "trust_improvement": self.trust_level - self.human.initial_trust,
        }

## Initialize the partnership

The profile values below are fixed inputs to the rule-based example. Risk tolerance, relationship focus, and initial trust are scores on a 0-to-1 scale; their interpretation is encoded by the human decision rules and trust update.

In [ ]:
human_expert = HumanExpert(
    name="Sarah Chen",
    experience_years=12,
    expertise_areas=[
        "contract_design",
        "relationship_management",
        "risk_assessment",
    ],
    risk_tolerance=0.4,
    relationship_focus=0.7,
    initial_trust=0.5,
)

ai_system = ExplainableAI(model_complexity="advanced")
partnership = SymbioticPartnership(human_expert, ai_system)

print("=== Human-AI Symbiotic Partnership Initialized ===")
print(
    f"Human Expert: {human_expert.name} "
    f"({human_expert.experience_years} years of experience)"
)
print(f"Initial Trust Level: {partnership.trust_level:.2f}")
print(f"AI Model Complexity Setting: {ai_system.model_complexity}")

## Generate demand scenarios

The generator combines a base demand level, a small linear trend, a sinusoidal seasonal component, and normally distributed noise. The fixed seed makes the generated example reproducible. The volatile, growing, and declining scenarios are transformations of the same base series, which makes their differences easier to interpret.

In [ ]:
def generate_complex_demand(periods: int) -> np.ndarray:
    """Generate demand with trend, seasonality, and seeded random noise."""
    if periods < 2:
        raise ValueError("At least two periods are required.")

    rng = np.random.RandomState(42)
    demand = np.zeros(periods, dtype=float)
    base_demand = 1000.0

    for t in range(periods):
        trend = 5.0 * t / periods
        seasonal = 200.0 * np.sin(2.0 * np.pi * t / 12.0)
        noise = rng.normal(0.0, 150.0)
        demand[t] = max(100.0, base_demand + trend + seasonal + noise)

    return demand


base_demand = generate_complex_demand(52)
scenario_rng = np.random.RandomState(42)
scenarios = {
    "stable": base_demand.copy(),
    "volatile": base_demand
    * scenario_rng.uniform(0.8, 1.2, len(base_demand)),
    "growing": base_demand * np.linspace(1.0, 1.3, len(base_demand)),
    "declining": base_demand * np.linspace(1.0, 0.8, len(base_demand)),
}

print(f"Generated {len(scenarios)} demand scenarios")
for scenario_name, demand in scenarios.items():
    print(
        f"{scenario_name.title()}: "
        f"Mean={np.mean(demand):.1f}, Std={np.std(demand):.1f}"
    )

## Conduct collaborative sessions

Each scenario is passed through the same ordered workflow. The context values are fixed across sessions so that the demand scenario and the partnership's accumulated AI preferences are the main changing inputs. The trust score is updated according to acceptance and confidence thresholds, not according to measured real-world success.

In [ ]:
print("=== Running Collaborative Sessions ===")

context = {
    "partner_history": {
        "reliability": 0.75,
        "flexibility": 0.65,
        "communication_quality": 0.8,
    },
    "data_quality": 0.85,
    "market_conditions": "moderate",
}

session_results = []
for i, (scenario_name, demand_data) in enumerate(scenarios.items(), start=1):
    print(f"\n--- Session {i}: {scenario_name.title()} Scenario ---")
    scenario_context = context.copy()
    scenario_context["scenario_name"] = scenario_name

    session_result = partnership.conduct_collaborative_session(
        demand_data, scenario_context
    )
    session_results.append(session_result)

    ai_recommendation = session_result["ai_recommendation"]
    human_evaluation = session_result["human_evaluation"]
    performance = session_result["performance"]
    print(f"AI Confidence: {ai_recommendation['confidence']:.3f}")
    print(f"Human Acceptance: {human_evaluation['acceptance_score']:.3f}")
    print(
        f"Trust Level: {session_result['trust_before']:.3f} "
        f"to {session_result['trust_after']:.3f}"
    )
    print(f"Bullwhip Ratio: {performance['bullwhip_ratio']:.3f}")
    print(f"Net Profit: ${performance['net_profit']:,.2f}")

## Summarize partnership outcomes

The summary aggregates the session history. Average acceptance describes agreement under the defined score; average bullwhip ratio and profit are outputs of the simplified performance equations, not independently validated operational measures. Preference factors show how the AI rule has incorporated substantial human modifications.

In [ ]:
summary = partnership.get_partnership_summary()

print("=== Partnership Summary ===")
print(f"Total Sessions: {summary['total_sessions']}")
print(f"Initial Trust: {human_expert.initial_trust:.3f}")
print(f"Final Trust: {summary['final_trust_level']:.3f}")
print(f"Trust Change: {summary['trust_improvement']:+.3f}")
print(f"Average Acceptance: {summary['average_acceptance']:.3f}")
print(f"Average Bullwhip Ratio: {summary['average_bullwhip_ratio']:.3f}")
print(f"Average Simulated Profit: ${summary['average_profit']:,.2f}")

print("\n=== AI Preference Factors ===")
if partnership.ai.human_preferences:
    for parameter, factor in partnership.ai.human_preferences.items():
        print(f"{parameter}: {factor:.3f}")
else:
    print("No preference factors met the learning rule's update conditions.")

## Visualize session results

The four panels show trust over the initial state and sessions, acceptance alongside AI confidence, average human parameter changes, and the performance proxies by scenario. In the parameter-change panel, green bars indicate positive changes and red bars indicate negative changes; the horizontal zero line indicates no average change. The final panel uses separate vertical axes because bullwhip ratio and profit have different units and scales.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(
    "Human-AI Partnership: Illustrative Session Analysis",
    fontsize=16,
    fontweight="bold",
)

trust_evolution = summary["trust_evolution"]
trust_sessions = np.arange(len(trust_evolution))
axes[0, 0].plot(
    trust_sessions,
    trust_evolution,
    "go-",
    linewidth=2,
    markersize=8,
    label="Trust score",
)
axes[0, 0].axhline(
    y=human_expert.initial_trust,
    color="red",
    linestyle="--",
    alpha=0.7,
    label="Initial trust",
)
axes[0, 0].set_title("Trust Score Across Sessions", fontweight="bold")
axes[0, 0].set_xlabel("Session index (0 is the initial state)")
axes[0, 0].set_ylabel("Trust score (0 to 1)")
axes[0, 0].set_ylim(0, 1)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

scenario_names = list(scenarios.keys())
acceptance_scores = [
    result["human_evaluation"]["acceptance_score"]
    for result in session_results
]
ai_confidences = [
    result["ai_recommendation"]["confidence"]
    for result in session_results
]
x_positions = np.arange(len(scenario_names))
bar_width = 0.35
axes[0, 1].bar(
    x_positions - bar_width / 2,
    acceptance_scores,
    bar_width,
    label="Human acceptance",
    color="lightblue",
    alpha=0.8,
)
axes[0, 1].bar(
    x_positions + bar_width / 2,
    ai_confidences,
    bar_width,
    label="AI confidence",
    color="lightcoral",
    alpha=0.8,
)
axes[0, 1].set_title("Acceptance and Confidence by Scenario", fontweight="bold")
axes[0, 1].set_ylabel("Score (0 to 1)")
axes[0, 1].set_xticks(x_positions)
axes[0, 1].set_xticklabels([name.title() for name in scenario_names], rotation=30)
axes[0, 1].set_ylim(0, 1)
axes[0, 1].grid(True, alpha=0.3, axis="y")
axes[0, 1].legend()

parameter_changes = {}
for result in session_results:
    ai_params = result["ai_recommendation"]["parameters"]
    human_params = result["human_evaluation"]["parameters"]
    for parameter, ai_value in ai_params.items():
        parameter_changes.setdefault(parameter, []).append(
            (human_params[parameter] - ai_value) / ai_value * 100.0
        )

parameter_names = list(parameter_changes)
average_changes = [
    float(np.mean(parameter_changes[parameter]))
    for parameter in parameter_names
]
bar_colors = [
    "green" if change > 0 else "red" if change < 0 else "gray"
    for change in average_changes
]
axes[1, 0].bar(
    np.arange(len(parameter_names)),
    average_changes,
    color=bar_colors,
    alpha=0.7,
)
axes[1, 0].set_title("Average Human Parameter Changes", fontweight="bold")
axes[1, 0].set_ylabel("Average change relative to AI recommendation (%)")
axes[1, 0].set_xticks(np.arange(len(parameter_names)))
axes[1, 0].set_xticklabels(
    [parameter.replace("_", "\n") for parameter in parameter_names],
    rotation=30,
)
axes[1, 0].grid(True, alpha=0.3, axis="y")
axes[1, 0].axhline(y=0, color="black", linestyle="-", alpha=0.5)
axes[1, 0].plot([], [], color="green", label="Increase")
axes[1, 0].plot([], [], color="red", label="Decrease")
axes[1, 0].legend()

bullwhip_ratios = [
    result["performance"]["bullwhip_ratio"] for result in session_results
]
profits_in_thousands = [
    result["performance"]["net_profit"] / 1000.0
    for result in session_results
]
profit_axis = axes[1, 1].twinx()
bullwhip_line = axes[1, 1].plot(
    x_positions,
    bullwhip_ratios,
    "ro-",
    linewidth=2,
    markersize=8,
    label="Bullwhip ratio",
)
profit_line = profit_axis.plot(
    x_positions,
    profits_in_thousands,
    "bs-",
    linewidth=2,
    markersize=8,
    label="Simulated net profit",
)
axes[1, 1].set_title("Performance Proxies by Scenario", fontweight="bold")
axes[1, 1].set_xlabel("Demand scenario")
axes[1, 1].set_ylabel("Bullwhip ratio", color="red")
axes[1, 1].tick_params(axis="y", labelcolor="red")
profit_axis.set_ylabel("Simulated net profit ($ thousands)", color="blue")
profit_axis.tick_params(axis="y", labelcolor="blue")
axes[1, 1].set_xticks(x_positions)
axes[1, 1].set_xticklabels([name.title() for name in scenario_names], rotation=30)
axes[1, 1].grid(True, alpha=0.3)
lines = bullwhip_line + profit_line
axes[1, 1].legend(lines, [line.get_label() for line in lines], loc="best")

fig.tight_layout()
plt.show()

## Compare the stated heuristic approaches

The AI-only approach uses the same demand-analysis and recommendation rules without human feedback. A separate AI instance ensures that its recommendations do not inherit preference factors learned during the collaborative sessions. The human-only approach uses a fixed illustrative parameter set. The symbiotic approach uses the recorded final human-reviewed contracts.

This is a descriptive comparison under one set of rules and scenarios, not a controlled empirical study. The approaches are not optimized against a common objective, and the human-only parameter set is a specified heuristic rather than a measured human decision process. Interpret the metrics as outputs of the simulation, not evidence that one approach is generally superior.

In [ ]:
print("=== Illustrative Heuristic Comparison ===")

ai_only_system = ExplainableAI(model_complexity=ai_system.model_complexity)
ai_only_performance = []
human_only_performance = []

human_only_params = {
    "revenue_sharing": 0.5,
    "flexibility_factor": 1.0,
    "safety_stock": 1.8,
    "info_sharing_bonus": 30,
    "lead_time_buffer": 0.7,
}

for scenario_name, demand_data in scenarios.items():
    demand_analysis = ai_only_system.analyze_demand_patterns(demand_data)
    ai_params, _, _ = ai_only_system.generate_contract_recommendation(
        demand_analysis, context
    )
    ai_only_performance.append(
        partnership._simulate_contract_performance(ai_params, demand_data)
    )
    human_only_performance.append(
        partnership._simulate_contract_performance(
            human_only_params, demand_data
        )
    )

symbiotic_bullwhip = float(
    np.mean([metric["bullwhip_ratio"] for metric in partnership.performance_metrics])
)
ai_only_bullwhip = float(
    np.mean([metric["bullwhip_ratio"] for metric in ai_only_performance])
)
human_only_bullwhip = float(
    np.mean([metric["bullwhip_ratio"] for metric in human_only_performance])
)

symbiotic_profit = float(
    np.mean([metric["net_profit"] for metric in partnership.performance_metrics])
)
ai_only_profit = float(
    np.mean([metric["net_profit"] for metric in ai_only_performance])
)
human_only_profit = float(
    np.mean([metric["net_profit"] for metric in human_only_performance])
)

print("Average bullwhip ratio:")
print(f"  Human-only heuristic: {human_only_bullwhip:.3f}")
print(f"  AI-only heuristic: {ai_only_bullwhip:.3f}")
print(f"  Symbiotic workflow: {symbiotic_bullwhip:.3f}")
print("\nAverage simulated net profit:")
print(f"  Human-only heuristic: ${human_only_profit:,.2f}")
print(f"  AI-only heuristic: ${ai_only_profit:,.2f}")
print(f"  Symbiotic workflow: ${symbiotic_profit:,.2f}")

## Interpreting the results

The generated outputs answer several different questions:

- **Trust:** Did the threshold-based update increase, decrease, or leave the trust score unchanged? A change reflects the model's confidence-and-acceptance rules, not an independently observed measure of trust.
- **Acceptance:** How much did the human-modified parameters differ from the AI recommendation, adjusted by AI confidence? A higher score indicates greater agreement under this definition.
- **Learning:** Which preference factors changed under the specified acceptance and modification thresholds? These factors affect later AI recommendations through multiplication.
- **Performance:** How did each parameter set affect the simplified bullwhip, cost, service-level, and profit calculations? These indicators are internally defined teaching proxies.
- **Comparison:** How do the three stated heuristics differ on the generated scenarios? The result is specific to this model and does not establish a general performance advantage.

The workflow illustrates how explicit rules can connect data analysis, human review, feedback, and performance reporting. Its main limitations are the stylized demand and cost model, hand-set decision rules, fixed scenario context, and absence of empirical calibration. Further work within this modeling framework could examine additional demand scenarios, partner contexts, or contract parameters, while keeping the assumptions and interpretation of the resulting metrics explicit.

### When this framework may be useful

This type of workflow can be useful as a teaching prototype when decisions require both quantitative summaries and documented human review, especially when preferences and trust are expected to evolve over repeated interactions. Real deployment would require validated data, clearly defined units, stakeholder review, and evaluation against operational outcomes.
